# Notebook for plotting additional Rufu & Canup (2020) evolution cases: Figures S2 to S4 
## Preamble
### Load packages required

In [1]:
#SJL 3/2023
#Script to plot a comparison of the the change in surface length for example tidal evolution scenarios

###########################################################
###########################################################
###########################################################
import numpy as np
import scipy as sp
import sys
import os
import struct
from scipy import constants as const

#package to use wildcards 
import fnmatch

import csv
import time as tclock

from scipy.interpolate import LinearNDInterpolator
from scipy.interpolate import griddata

#plotting packages
import matplotlib as mpl
import matplotlib.pyplot as plt
import pylab
import matplotlib.cm as cm
from matplotlib import gridspec


cwd = os.getcwd()
print(cwd)
if sys.platform== 'darwin':
    sys.path.insert(0, cwd+'/Support_scipts')
    print(cwd+'/Support_scripts')
elif (sys.platform== 'win32') | (sys.platform== 'win64'):
    sys.path.insert(0,cwd+"\\Support_scipts")
    
#HERCULES_structures
from HERCULES_structures import *
from surface_size_calc import *
from HERCULES_random_planet_database_structure_1D import *

#functions for calculating non-evenly spaced numerical differentials
from gradients import *

#import colormaps
import colormaps as cmaps
import matplotlib.cm as cm

import svglib.svglib as svglib
svglib.register_font('helvetica', './Helvetica.ttc')

/Users/vq21447/Documents/Lock_2026_SI
/Users/vq21447/Documents/Lock_2026_SI/Support_scripts
CHECK THE LOCATION OF ODYSSEY BACKUP
CHECK THE LOCATION OF ODYSSEY BACKUP


('helvetica', True)

### Define constants

In [ ]:
########################################################################################
########################################################################################
########################################################################################
#CONSTANTS
MEarth=5.972E24
LEM=3.5E34
REarth=6.371E6
MMoon=7.34767309E22

aMoon=0.3844E9
aCassini=30*REarth
aRoche=2.9*REarth

#for HERCULES
MEarth_H=5.9879648E24
LEM_H=3.53E34

### Set parameters and select scenarios to plot

In [ ]:
########################################################################################
########################################################################################
########################################################################################
#PARAMS
#info for HERCULES arrays
Hdir='Earth_correct_params_S3.20c'
Hname='Earth_correct_params_S3.20c'

#directory to put plots
plot_dir='Rufu&Canup_plots'
if os.path.isdir(plot_dir)==False:
    os.mkdir(plot_dir)

#number of contor points
Ncont=100 #100dd
Nskip=50 #15

#Earth's moment of inertia used to convert to AM
C_Earth=0.335

#Give data files
#0: A=5, tau=49E5, Q/k=406
#1: A=500, tau=200E5, Q/k=406
#2: A=10000, tau=1500E5, Q/k=406
dir='Data_repo/Rufu&Canup_2020_quasi_resonance/ResultsEvection_2'
file_names=['EvolutionA5_Tau49.0e5_Phi0.txt',
            'EvolutionA500_Tau200.0e5_Phi0.txt',
            'EvolutionA10000_Tau1500.0e5_Phi0.txt']

plot_names=['FigureS2','FigureS3','FigureS4']

#output file root
data_output_file_root='Rufu&Canup_surf_change'

#directory to find procesed length data
data_dir='Data_repo/Rufu&Canup_2020_quasi_resonance'

## Main script
### Load in HERCULES database

In [ ]:
########################################################################################
########################################################################################
########################################################################################
#MAIN

########################################################################################
#read in the database

Hdatabase=HERCULES_random_planet_database_1D()
Hdatabase.make_array(Hdir,Hname)
Hdatabase.initialize_interpolation([0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1],\
                                   [0,0,[0],0,0,0,0,[0],[0],[0],[0],[0],[0],[0],[0],[0],[0]], flag_extrap=1)

#extract the latitudes for each Mu point
Nmu=Hdatabase.parr[0].Nmu
lat=np.arccos(Hdatabase.parr[0].layers[0].mu)*180/np.pi

### Loop over each scenario, load in the 

In [ ]:
for flag_data in np.arange(3):
    print(plot_names[flag_data],'\t',file_names[flag_data])
    fdata=open(dir+'/'+file_names[flag_data])
    
    reader = csv.reader(fdata, delimiter="\t", skipinitialspace=True)
    temp = list(reader)
    
    Ctime=[]
    CkQ=[]
    Ca=[]
    Comg=[]
    Comg_moon=[]
    Ce=[]
    Cphi=[]
    
    
    for i in np.arange(len(temp)):
        if i==0:
            Aset=temp[i][0]
            Aset=float(Aset[6:])

        elif i==1:
            tau_set=temp[i][0]
            tau_set=float(tau_set[13:])

        elif (i>3)&(np.size(temp[i])>3):

            Ctime.append(temp[i][0]) #time
            Comg.append(temp[i][1]) #rotation rate of Earth
            Comg_moon.append(temp[i][2]) #rotation rate of Moon
            Ce.append(temp[i][3]) #eccentricity
            Ca.append(temp[i][4]) #semi-major axis
            Cphi.append(temp[i][5]) #Phi parameter related to tidal lag
        
            
    #convert to useful type and units
    omg_norm=np.sqrt(const.G*MEarth/(REarth**3))
    Ctime=np.asarray(Ctime, dtype=np.float64)*1E4
    Ca=np.asarray(Ca, dtype=np.float64)*REarth
    Ce=np.asarray(Ce, dtype=np.float64)
    Comg=np.asarray(Comg, dtype=np.float64)*omg_norm
    Comg_moon=np.asarray(Comg, dtype=np.float64)*omg_norm
    
    #calculate the AM and de-normalize
    CL_tot=Comg/omg_norm+1.07E-3*Comg_moon/omg_norm+0.0367*np.sqrt(Ca/REarth*(1-Ce**2))
    CL_tot=CL_tot*C_Earth*MEarth*REarth**2*omg_norm
    
    CL=Comg*C_Earth*MEarth*REarth**2


    #define an output file
    data_output_file=data_dir+'/'+data_output_file_root+'_A_'+str(Aset)+'_tau_'+str(tau_set/1E5)+'E5.bin'
    data_output_file_int=data_dir+'/'+data_output_file_root+'_max_integrated_A_'+str(Aset)+'_tau_'+str(tau_set/1E5)+'E5.bin'

    #read in the max deformation data 
    dataf_int = open(data_output_file_int, "rb")
    
    #read in the file as one massive array
    ndata_per_step=6
    data = np.fromfile(dataf_int, dtype=np.float64, count=-1)
    dataf_int.close()
    
    #work out how many time steps we have
    temp=np.size(data)*1.0/(1.0*ndata_per_step)
    if abs(temp-int(temp))<(1E-12):
        Ntstep=int(temp)
    else:
        print("ERROR IN READ",'\n',"Not complete number of steps or incorrect number of print params",'\n',"EXITING")
    
    #reshape array so that each row is a timestep
    data=data.reshape(Ntstep,ndata_per_step)
    
    steps=data[:,0].astype(int)
    time=data[:,1]
    ddl_lon_dt_max=data[:,2]
    ddl_lon_dt_min=data[:,3]
    ddl_lat_dt_min=data[:,4]
    ddl_lat_dt_max=data[:,5]



    #create a dummy figure figure first to avoid plotting issues
    fig = plt.figure(figsize=(3.9,6.5))
    font = {
    'family' : 'Helvetica',
            'weight' : 'normal',
            'size'   : 8}
    mpl.rc('font', **font)
    gs = gridspec.GridSpec(2, 1, height_ratios=[0.64,1])
    ax=plt.subplot(gs[0])
    ax1=plt.subplot(gs[1])
    fig.tight_layout()
    plt.savefig(plot_names[flag_data]+'.pdf')
    mpl.pyplot.close(fig)
    
    
    #initialise the figure
    fig = plt.figure(figsize=(3.5,8.0))
    
    gs0 = gridspec.GridSpec(3, 1,
                            width_ratios=[1],
                            height_ratios=[1,1,1],
                            )
    
    gs000 = gridspec.GridSpecFromSubplotSpec(2, 1, 
                                        subplot_spec=gs0[0],
                                        height_ratios=[1,1],
    #                                         hspace=0.05,
    #                                         wspace=0.05
                                        )
    
    
    ax=[[],[],[]]
    ax[0].append(plt.subplot(gs0[1]))
    
    ax[1].append(plt.subplot(gs0[2], sharex=ax[0][0]))
    
    ax[2].append(plt.subplot(gs000[0], sharex=ax[0][0]))
    ax[2].append(plt.subplot(gs000[1], sharex=ax[0][0]))
    
    font = {
    'family' : 'Helvetica',
            'weight' : 'normal',
            'size'   : 8}
    mpl.rc('font', **font)
    
    col=cmaps.parula([0.15,0.85])
    
    
    ax[2][0].plot(Ctime/1E6, Ca/REarth, 'k-', linewidth=1.5)
    ax[2][1].plot(Ctime/1E6, CL/LEM, 'k-', linewidth=1.5)
    
    
    if flag_data==0:
        
        ax[2][0].set_ylim([3,18])
    
    elif flag_data==1:
    
        ax[2][0].set_ylim([3,11])
        ax[2][1].set_ylim([0.5,2.2])
    
    elif flag_data==2:
    
        ax[2][0].set_ylim([3,12])
    
    
    ind_max=-1
    
    
    ax[0][0].plot([0,time[ind_max]/1E6],[8E-3,8E-3],'--', color=col[0],linewidth=1.5) #slow mid ocean ridge
    ax[0][0].plot([0,time[ind_max]/1E6],[15E-3,15E-3],'--', color=col[1],linewidth=1.5) #slow subduction
    ax[0][0].plot([0,time[ind_max]/1E6],[50E-3,50E-3],':', color=col[0],linewidth=1.5) #average mid-ocean ridge
    ax[0][0].plot([0,time[ind_max]/1E6],[60E-3,60E-3],':', color=col[1],linewidth=1.5) #avererage subduction
    #ax[0][0].plot([0,t[ind_max]/1E6],[4E-4,4E-4],'b--') 
    
    ax[1][0].plot([0,time[ind_max]/1E6],[60E-3,60E-3],':', color=col[1],linewidth=1.5, label='Average subduction') #avererage subduction
    ax[1][0].plot([0,time[ind_max]/1E6],[50E-3,50E-3],':', color=col[0],linewidth=1.5, label='Average ridge') #average mid-ocean ridge
    ax[1][0].plot([0,time[ind_max]/1E6],[15E-3,15E-3],'--', color=col[1],linewidth=1.5, label='Slow subduction') #slow subduction
    ax[1][0].plot([0,time[ind_max]/1E6],[8E-3,8E-3],'--', color=col[0],linewidth=1.5, label='Slow ridge') #slow mid ocean ridge
    
    ax[0][0].plot(time/1E6, np.absolute(ddl_lat_dt_max),'-', color=col[0],linewidth=1.5,label='Extension (polar)')
    ax[0][0].plot(time/1E6, np.absolute(ddl_lat_dt_min),'-', color=col[1],linewidth=1.5,label='Convergence (eq.)')
    
    
    ax[1][0].plot(time/1E6, np.absolute(ddl_lon_dt_max),'-', color=col[0],linewidth=1.5)
    ax[1][0].plot(time/1E6, np.absolute(ddl_lon_dt_min),'-', color=col[1],linewidth=1.5)
    
    # ax[0][2].plot(t/1E6, np.absolute(ddA_dt_max)*const.year*(np.pi/180.0)**2,'r-')
    # ax[0][2].plot(t/1E6, np.absolute(ddA_dt_min)*const.year*(np.pi/180.0)**2,'k-')
    
    
    
    ax[0][0].set_yscale('log')
    ax[1][0].set_yscale('log')
    
    ax[2][0].set_ylabel(r"$a_{\rm Moon}$ [$R_{\rm Earth}$]")
    ax[2][1].set_ylabel(r"AM Earth [$L_{\rm EM}$]")
    
    ax[0][0].set_ylabel('Integrated deformation rate [m yr$^{-1}$]')
    ax[1][0].set_ylabel('Integrated deformation rate [m yr$^{-1}$]')
    
    ax[1][0].set_xlabel('Time [Myrs]')
    
    plt.setp( ax[0][0].get_xticklabels(), visible=False)
    plt.setp( ax[2][0].get_xticklabels(), visible=False)
    plt.setp( ax[2][1].get_xticklabels(), visible=False)
    
    ax[2][0].tick_params(direction="in", top=True, right=True)
    ax[2][1].tick_params(direction="in", top=True, right=True)
    ax[0][0].tick_params(direction="in", top=True)
    ax[1][0].tick_params(direction="in", top=True)
    
    if flag_data==0:
        ax[0][0].legend(frameon=False, handlelength=1.0, loc='center right')#,bbox_to_anchor=(1.0, 0.92))
        ax[1][0].legend(frameon=False, handlelength=1.9, loc='center right',bbox_to_anchor=(1.0, 0.6))
        
        ax[0][0].text(0.7, 0.94, 'c: Latitudinal', horizontalalignment='left',verticalalignment='top', fontsize=8,transform=ax[0][0].transAxes, color='k')
        ax[1][0].text(0.7, 0.94, 'd: Longitudinal', horizontalalignment='left',verticalalignment='top', fontsize=8,transform=ax[1][0].transAxes, color='k')
           
    
    elif flag_data==1:
        ax[0][0].legend(frameon=False, handlelength=1.0, loc='center right',bbox_to_anchor=(1.0, 0.7))
        ax[1][0].legend(frameon=False, handlelength=1.9, loc='upper left',bbox_to_anchor=(0.06, 0.95))
        
        
        ax[0][0].text(0.7, 0.94, 'c: Latitudinal', horizontalalignment='left',verticalalignment='top', fontsize=8,transform=ax[0][0].transAxes, color='k')
        ax[1][0].text(0.7, 0.94, 'd: Longitudinal', horizontalalignment='left',verticalalignment='top', fontsize=8,transform=ax[1][0].transAxes, color='k')
           
    
    else:
        
        ax[0][0].legend(frameon=False, handlelength=1.0, loc='upper right')
        ax[1][0].legend(frameon=False, handlelength=1.9, loc='upper right')#,bbox_to_anchor=(0.98, 1.686))
        
        
        ax[0][0].text(0.05, 0.05, 'c: Latitudinal', horizontalalignment='left',verticalalignment='bottom', fontsize=8,transform=ax[0][0].transAxes, color='k')
        ax[1][0].text(0.05, 0.05, 'd: Longitudinal', horizontalalignment='left',verticalalignment='bottom', fontsize=8,transform=ax[1][0].transAxes, color='k')
           
    
    
    ax[2][0].text(0.05, 0.94, 'a', horizontalalignment='left',verticalalignment='top', fontsize=8,transform=ax[2][0].transAxes, color='k')
    ax[2][1].text(0.05, 0.06, 'b', horizontalalignment='left',verticalalignment='bottom', fontsize=8,transform=ax[2][1].transAxes, color='k')
      
    
    fig.tight_layout()
    plt.savefig(plot_names[flag_data]+'.pdf')
    
    print('\t done')